In [1]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import numpy as np
from sklearn.model_selection import train_test_split

load_dotenv(find_dotenv())

engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host="localhost",
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME"),
))

df = pd.read_parquet('../data/application_clean.parquet')
df

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year,credit_over_goods,fee_band,age_years,social_def_rate_30
0,100002,1,0,1,0,1,0,202500.0,406597.5,24700.5,...,0.0,0.0,0.0,0.0,0.0,1.0,1.158397,fees_added,25.903338,1.0
1,100003,0,0,0,0,0,0,270000.0,1293502.5,35698.5,...,0.0,0.0,0.0,0.0,0.0,0.0,1.145199,fees_added,45.901011,0.0
2,100004,0,1,1,1,1,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,no_fees,52.146177,NaN
3,100006,0,0,0,0,1,0,135000.0,312682.5,29686.5,...,NaN,NaN,NaN,NaN,NaN,NaN,1.052803,fees_added,52.033923,0.0
4,100007,0,0,1,0,1,0,121500.0,513000.0,21865.5,...,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,no_fees,54.571962,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307502,456251,0,0,1,0,0,0,157500.0,254700.0,27558.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.132000,fees_added,25.536459,NaN
307503,456252,0,0,0,0,1,0,72000.0,269550.0,12001.5,...,NaN,NaN,NaN,NaN,NaN,NaN,1.198000,fees_added,56.880018,NaN
307504,456253,0,0,0,0,1,0,153000.0,677664.0,29979.0,...,1.0,0.0,0.0,1.0,0.0,1.0,1.158400,fees_added,40.975516,0.0
307505,456254,1,0,0,0,1,0,171000.0,370107.0,20205.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.158394,fees_added,32.748106,NaN


In [2]:
ids = df['sk_id_curr']
X = df.drop(columns = ['target', 'sk_id_curr'])
y = df['target']

print(f'ID data:\n{ids.head()}\n')
print(f'Train data:\n{X.head()}\n')
print(f'Target data:\n{y.head()}')

#Temporary will be splitten again to create a seperate data to build an EV threshold model on unseen data.
#Ids are kept in seperate data(while being connected to our main data), this is done to have the option later on to connected the datasets
#using clients IDS
X_temporary, X_test, y_temporary, y_test, ids_temporary, ids_test = train_test_split(X, y, ids, test_size = 0.2, random_state = 42)
X_train, X_evModel, y_train, y_evModel, ids_train, ids_evModel = train_test_split(X_temporary, y_temporary, ids_temporary, test_size = 0.25, random_state = 42)

ID data:
0    100002
1    100003
2    100004
3    100006
4    100007
Name: sk_id_curr, dtype: int64

Train data:
   name_contract_type  code_gender  flag_own_car  flag_own_realty  \
0                   0            1             0                1   
1                   0            0             0                0   
2                   1            1             1                1   
3                   0            0             0                1   
4                   0            1             0                1   

   cnt_children  amt_income_total  amt_credit  amt_annuity  amt_goods_price  \
0             0          202500.0    406597.5      24700.5         351000.0   
1             0          270000.0   1293502.5      35698.5        1129500.0   
2             0           67500.0    135000.0       6750.0         135000.0   
3             0          135000.0    312682.5      29686.5         297000.0   
4             0          121500.0    513000.0      21865.5         513000.0  

In [3]:
print(f'60% Train:{X_train.shape, y_train.shape, ids_train.shape} \n20% Test:{X_test.shape, y_test.shape, ids_test.shape} \n20% evModel:{X_evModel.shape, y_evModel.shape}')

60% Train:((184503, 84), (184503,), (184503,)) 
20% Test:((61502, 84), (61502,), (61502,)) 
20% evModel:((61502, 84), (61502,))


In [4]:
#Splitting the learning data into two datas, one represeting numerical data and the other categorical data.
#It is required for this dataset because categorical and numerical data require different treatment.

categorical_cols = X.select_dtypes('object').columns.tolist()
numerical_cols = X.select_dtypes('number').columns.tolist()

print(len(categorical_cols), "categorical")
print(len(numerical_cols), "numeric")
print(f'target in X: {'target' in X.columns} | sk_id_curr in X:{'sk_id_curr' in X.columns}')
print(f'Categorical + Numerical columns sum:{len(categorical_cols) + len(numerical_cols)}, Learning data columns sum:{X.shape[1]}')

13 categorical
71 numeric
target in X: False | sk_id_curr in X:False
Categorical + Numerical columns sum:84, Learning data columns sum:84


In [24]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imputer = SimpleImputer(missing_values=np.nan, strategy='median')
X_train_imputed = imputer.fit_transform(X_train[numerical_cols])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)

print(X_train_scaled.shape, X_train_scaled.mean(axis=0)[:3].round(3))

(184503, 71) [ 0. -0. -0.]
